In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# Portable project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
IMAGE_DIR = PROJECT_ROOT / "images"

for directory in (OUTPUT_DIR, MODEL_DIR, IMAGE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

columns_path = DATA_DIR / "census-bureau.columns"
if not columns_path.exists():
    raise FileNotFoundError(
        f"Missing {columns_path}. Place the Census Income column file in data/ before running the notebook."
    )

cols_df = pd.read_csv(columns_path, header=None)
columns = [str(x).strip() for x in cols_df[0].tolist()]
print(f"Columns loaded successfully: {len(columns)}")

In [ ]:
data_path = DATA_DIR / "census-bureau.data"
if not data_path.exists():
    raise FileNotFoundError(
        f"Missing {data_path}. Place the Census Income data file in data/ before running the notebook."
    )

df = pd.read_csv(data_path, names=columns, na_values=[" ?", "?"])
print("Data loaded successfully.")
print("Data shape:", df.shape)

In [ ]:
df.columns

In [ ]:
df['income_label'] = df['label'].astype(str).str.strip().apply(lambda x: 1 if '50000+.' in x else 0)

df['income_label'].value_counts() 

In [ ]:
X = df.drop(columns=["label", "income_label", "year"], errors="ignore").copy()
y = df["income_label"].copy()

In [ ]:
# Remove exact duplicate observations while preserving any rows that share features but have different labels.
duplicate_mask = pd.concat([X, y.rename("income_label")], axis=1).duplicated()
X = X.loc[~duplicate_mask].copy()
y = y.loc[~duplicate_mask].copy()
df = df.loc[~duplicate_mask].copy()
print(f"Removed {duplicate_mask.sum()} exact duplicate observations.")

In [ ]:
from sklearn.model_selection import train_test_split

# Split before fitting preprocessing steps to avoid train/test leakage.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

numeric_cols = X_train_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train_raw.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
], verbose_feature_names_out=False)

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out()

X_train = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test_raw.index)
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

In [ ]:
# Clip numeric outliers using bounds learned from the training set only.
outlier_bounds = {}
for col in numeric_cols:
    if col not in X_train.columns:
        continue
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_bounds[col] = (lower, upper)
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

In [ ]:
# Select features using training data only.
train_correlations = X_train.corrwith(y_train).sort_values(ascending=False)
correlation_threshold = 0.01
selected_features = train_correlations[train_correlations.abs() > correlation_threshold].index.tolist()

X_train = X_train[selected_features]
X_test = X_test[selected_features]
print(f"Selected {len(selected_features)} features using |correlation| > {correlation_threshold}.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
print(f"Training accuracy: {rf_model.score(X_train, y_train):.4f}")
print(f"Test accuracy: {rf_model.score(X_test, y_test):.4f}")

## Probability-threshold analysis

The classifier is also evaluated at a 0.7 probability threshold to inspect the precision/recall trade-off for higher-confidence positive predictions. This is an analysis choice, not a claim that 0.7 is universally optimal.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

y_probs = rf_model.predict_proba(X_test)[:, 1]
probability_threshold = 0.7
y_pred_strict = (y_probs >= probability_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_strict)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["<50K", ">50K"])
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax)
plt.title("Confusion Matrix at 0.7 Probability Threshold")
plt.tight_layout()
plt.show()
print(classification_report(y_test, y_pred_strict))

In [ ]:
# Feature importance from the fitted Random Forest.
feature_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_,
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x="Importance", y="Feature", data=feature_importance_df.head(15))
plt.title("Top 15 Random Forest Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

print(f"ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}")
print(f"Average precision: {average_precision_score(y_test, y_probs):.4f}")

In [ ]:
# Persist the fitted classifier together with preprocessing metadata required for inference.
model_data = {
    "preprocessor": preprocessor,
    "outlier_bounds": outlier_bounds,
    "classifier": rf_model,
    "features": selected_features,
    "threshold": probability_threshold,
}
classification_model_path = MODEL_DIR / "census_income_final_model.pkl"
joblib.dump(model_data, classification_model_path)
print(f"Classification model saved to {classification_model_path}")

## Customer segmentation

The segmentation portion keeps the original K-Means approach. Selected demographic and income-related variables are encoded, standardized, and grouped into five clusters. The elbow plot is used as a diagnostic for the selected cluster count.

In [ ]:
segment_cols = [
    "age",
    "education",
    "marital stat",
    "major occupation code",
    "sex",
    "weeks worked in year",
    "capital gains",
    "income_label",
]
df_cluster = df[segment_cols].copy()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

cluster_categorical_cols = df_cluster.select_dtypes(exclude=["number"]).columns.tolist()
cluster_numeric_cols = df_cluster.select_dtypes(include=["number"]).columns.tolist()

cluster_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
if cluster_categorical_cols:
    df_cluster[cluster_categorical_cols] = cluster_encoder.fit_transform(
        df_cluster[cluster_categorical_cols].astype(str)
    )

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)

In [ ]:
from sklearn.cluster import KMeans

wcss = []
for i in range(1, 11):
    candidate = KMeans(n_clusters=i, init="k-means++", random_state=42, n_init=10)
    candidate.fit(df_scaled)
    wcss.append(candidate.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of clusters")
plt.ylabel("WCSS")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "elbow_method.png", dpi=150)
plt.show()

In [ ]:
# Fit K-Means using the selected five-cluster solution.
kmeans = KMeans(n_clusters=5, init="k-means++", random_state=42, n_init=10)
df["Segment"] = kmeans.fit_predict(df_scaled)
print(df["Segment"].value_counts().sort_index())

In [ ]:
income_col = "income_label"

high_income_stats = (
    df.groupby("Segment")[income_col]
    .mean()
    .mul(100)
    .reset_index(name="High_Income_Percent")
)

numeric_profiles = df.groupby("Segment")[
    ["age", "capital gains", "weeks worked in year"]
].mean()
common_jobs = df.groupby("Segment")["major occupation code"].agg(
    lambda values: values.mode().iloc[0] if not values.mode().empty else "Unknown"
)

segment_report = pd.concat([numeric_profiles, common_jobs], axis=1).reset_index()
segment_report = segment_report.merge(high_income_stats, on="Segment")
segment_report = segment_report.sort_values("High_Income_Percent", ascending=False)
segment_report

In [ ]:
# Visualize cluster size and save the figure for the README/project report.
segment_counts = df["Segment"].value_counts().sort_index()
plt.figure(figsize=(8, 5))
sns.barplot(x=segment_counts.index, y=segment_counts.values)
plt.title("Customer Segment Sizes")
plt.xlabel("Segment")
plt.ylabel("Number of records")
plt.tight_layout()
segment_plot_path = IMAGE_DIR / "segment_sizes.png"
plt.savefig(segment_plot_path, dpi=150)
plt.show()

In [ ]:
segmentation_results_path = OUTPUT_DIR / "customer_segmentation_results.csv"
df.to_csv(segmentation_results_path, index=False)
print(f"Segmentation results saved to {segmentation_results_path}")

In [ ]:
# Persist the fitted clustering preprocessing objects and model.
clustering_assets = {
    "encoder": cluster_encoder if cluster_categorical_cols else None,
    "categorical_features": cluster_categorical_cols,
    "scaler": scaler,
    "model": kmeans,
    "features": segment_cols,
}
segmentation_model_path = MODEL_DIR / "customer_segmentation_model.pkl"
joblib.dump(clustering_assets, segmentation_model_path)
print(f"Segmentation model saved to {segmentation_model_path}")